# OpenAI API 基本用法

In [ ]:
# 金鑰從 Colab 左側欄的 🔑 Secrets 讀取（不在 Colab 時改讀環境變數），不寫在程式裡
import os

def get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

# 環境準備

如果這是你的第一份openAI專案（ Colab 環境沒有下面這些 package），輸入下方指令來安裝 package

In [28]:
!pip install --upgrade openai
!pip show openai

## 使用 requests 套件

In [ ]:
# import requests

# API_KEY = get_secret("OPENAI_API_KEY")

# response = requests.post(
#     'https://api.openai.com/v1/chat/completions',
#     headers={
#         'Content-Type': 'application/json',
#         'Authorization': f'Bearer {API_KEY}'
#     },
#     json={
#         'model': 'gpt-3.5-turbo',
#         'messages': [{"role": "user", "content": "用繁體中文說句話吧"}],
#     }
# )

# print(response.json())


In [ ]:
# response = requests.post(
#     'https://api.openai.com/v1/chat/completions',
#     headers={
#         'Content-Type': 'application/json',
#         'Authorization': f'Bearer {API_KEY}'
#     },
#     json={
#         'model': 'gpt-3.5-turbo',
#         'messages': [{"role": "user", "content": "今天最多人討論的話題是什麼？"}],
#     }
# )

# print(response.json())

## 使用openAI官方文件
https://platform.openai.com/docs/libraries/python-library

- user: 是使用者本人傳的訊息，所以一般使用者打在輸入匡的就是user這個role傳prompt給OpenAI。
- system: 這個role負責的則是類似系統設定的文字提示，例如跟OpenAI說他是一個工程師，接下來要請他用他的專業寫code；或是跟OpenAI說他是一個厲害的數學家，要請他用他的專業解題。
- assistant: 這個role代表的是AI本人，所以當一個訊息的role是assistant，代表這是AI講的話。例如AI回傳時會說role是assistant，還有另一個用途就是跟AI連續對話時，也可以用這個role去實現。

錯誤實可參考的解法：https://github.com/openai/openai-python/discussions/742

In [29]:
from openai import OpenAI

client = OpenAI(
  api_key=get_secret("OPENAI_API_KEY"),  # this is also the default, it can be omitted
)

###Hello OpenAI

回覆的message、token數等等的資訊。

In [5]:
completion = client.chat.completions.create(
  model="gpt-3.5-turbo",
  messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"}
  ]
)

print(completion)

ChatCompletion(id='chatcmpl-9Bd25JmeJ9za111NxPk7v7z1H5Jiy', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I help you today?', role='assistant', function_call=None, tool_calls=None))], created=1712558677, model='gpt-3.5-turbo-0125', object='chat.completion', system_fingerprint='fp_b28b39ffa8', usage=CompletionUsage(completion_tokens=9, prompt_tokens=19, total_tokens=28))


In [12]:
msg = input()
response = client.chat.completions.create(
  model="gpt-3.5-turbo",
  messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": msg}
  ]
)

print(response.choices[0].message.content)

講個笑話
當然！這是一個簡單的笑話：為什麼樹無法在考試前準備呢？因為它們都很樹懶！哈哈！希望這個笑話能讓您開心一下！有什麼可以我為您效助的嗎？


###Respone & Translate

回覆問題

In [ ]:
completion = client.chat.completions.create(
  model="gpt-3.5-turbo",
  messages=[
    {"role": "system", "content": "response by Traditional Chinese."},
    {"role": "user", "content": "who is 馬英九?"}
  ]
)

print(completion.choices[0].message.content)

馬英九是台灣的政治人物，曾擔任第12、13任中華民國總統，也是中國國民黨的黨員。他在2008年至2016年間擔任總統，期間實施了一系列政策和改革。在他的領導下，台灣與中國大陸之間的兩岸關係進一步緩和。在他的任期結束後，他退休並退出政壇。


以下對話延續了前面的例子，將OpenAI的回覆放到了messages中，這樣OpenAI才會知道之前自己講過了什麼，從而接續對話。

不過，對話越長，input長度越長，一次request要付的錢就會增加。

＃延續對話

In [ ]:
completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "system", "content": "response by Traditional Chinese."},
              {"role": "user", "content": "who is 馬英九?"},
              {"role": "assistant", "content": "馬英九是台灣的政治人物，曾擔任第12、13任中華民國總統，也是中國國民黨的黨員。他在2008年至2016年間擔任總統，期間實施了一系列政策和改革。在他的領導下，台灣與中國大陸之間的兩岸關係進一步緩和。在他的任期結束後，他退休並退出政壇。"},
              {"role": "user", "content": "what did he do before?"}
    ]
  )


print(completion.choices[0].message.content)

馬英九在擔任台灣總統之前，曾擔任過多個政府重要職位。他曾擔任過中華民國行政院副院長、第二、三、四、五、六、七、八屆立法委員、法務部長、內政部長等職位。在政治以外，馬英九也曾是一位學者和律師，並曾在美國留學研究法學。他一直以來都在政治界和學術界擁有著廣泛的影響力。


**＃關鍵字擷取**

In [ ]:
response = client.chat.completions.create(
  model="gpt-3.5-turbo",
  messages=[
    {
      "role": "system",
      "content": "You will be provided with a block of text, and your task is to extract a list of keywords from it."
    },
    {
      "role": "user",
      "content": "今天吃完飯後頭很痛，此外最近記憶力很差，一直會迷路或是做一樣的事"
    }
  ],
  temperature=0.5,
  max_tokens=64,
  top_p=1
)

print(response.choices[0].message.content)

Keywords:
- 頭痛
- 記憶力
- 迷路
- 做一樣的事


好難聚焦

In [ ]:
from openai import OpenAI

client = OpenAI(
  api_key=get_secret("OPENAI_API_KEY"),  # this is also the default, it can be omitted
)

text = input()

response = client.chat.completions.create(
  model="gpt-3.5-turbo",
  messages=[
    {
      "role": "system",
      "content": "You will be provided with a block of text, and your task is to extract a list of keywords from it"
    },
    {
      "role": "user",
      "content": text
    }
  ],
  temperature=0.5,
  max_tokens=64,
  top_p=1
)

print(response.choices[0].message.content)
answer=response.choices[0].message.content
response = client.chat.completions.create(
  model="gpt-3.5-turbo",
  messages=[
    {
      "role": "system",
      "content": "Your task is to use the provided keywords to organize the information into a well-formatted and segmented article. Focus on predicting potential symptoms and highlighting the aspects that need attention regarding those symptoms.limit in 50 words"
    },
    {
      "role": "user",
      "content": answer
    }
  ],
  temperature=0.5,
  max_tokens=64,
  top_p=1
)
print(response.choices[0].message.content)

今天吃完飯後頭很痛，此外最近記憶力很差，一直會迷路或是做一樣的事
1. 飯後頭痛
2. 記憶力差
3. 迷路
4. 做一樣的事
飯後頭痛可能是胃部問題引起，需關注飲食習慣。記憶力差可能暗示潛在健康問題，建議尋求


In [ ]:
from openai import OpenAI

client = OpenAI(
  api_key=get_secret("OPENAI_API_KEY"),  # this is also the default, it can be omitted
)

text = input()

response = client.chat.completions.create(
  model="gpt-3.5-turbo",
  messages=[
    {
      "role": "system",
      "content": "You will be provided with information regarding your chronic illness. Your task is to predict the potential risks and issues for the next episode."
    },
    {
      "role": "user",
      "content": text
    }
  ],
  temperature=0.5,
  max_tokens=64,
  top_p=1
)

print(response.choices[0].message.content)

我現在是糖尿病第一期
對於患有糖尿病的人來說，第一期通常指的是控制較為穩定且病情較輕微的階段。然而，


In [ ]:
from openai import OpenAI

client = OpenAI(
  api_key=get_secret("OPENAI_API_KEY"),  # this is also the default, it can be omitted
)

text = input()

response = client.chat.completions.create(
  model="gpt-3.5-turbo",
  messages=[
    {
      "role": "system",
      "content": "You will be provided with a tweet, and your task is to classify its sentiment as positive, neutral, or negative"
    },
    {
      "role": "user",
      "content": text
    }
  ],
  temperature=0.5,
  max_tokens=64,
  top_p=1
)

print(response.choices[0].message.content)

希望身體會好起來
Neutral


In [ ]:
response = client.chat.completions.create(
  model="gpt-3.5-turbo",
  messages=[
    {
      "role": "system",
      "content": "You will be provided with a block of text, and your task is to extract a list of keywords from it."
    },
    {
      "role": "user",
      "content": "今天吃完飯後頭很痛，此外最近記憶力很差，一直會迷路或是做一樣的事"
    }
  ],
  temperature=0.5,
  max_tokens=64,
  top_p=1
)

print(response.choices[0].message.content)

In [ ]:
# 初始化對話歷史
conversation_history = []

# 清除conversation_history
def clear_conversation_history():
    global conversation_history
    conversation_history = []

# 主程式碼
def get_response(text):
    global conversation_history
    # 檢查並處理重覆的內容
    if text in [msg['content'] for msg in conversation_history and msg['role'] == 'user']:
        return "您剛才已經問過這個問題了，請提供新的問題或主題。"

    # 使用 text 變數作為 answer 的值
    answer = text
    if answer is None or answer == "":
        return "請輸入有效的內容。"

    # 創建回覆請求
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=conversation_history + [
            {
              "role": "system",
              "content": "You will be provided with a block of text, and your task is to extract a list of keywords from it."
            },
            {"role": "user", "content": answer},
        ],
        temperature=0.25,
        max_tokens=400,
        top_p=1
    )

    # 更新對話歷史
    conversation_history.append({"role": "user", "content": answer})
    conversation_history.append({"role": "assistant", "content": response.choices[0].message.content})

    # 清除舊的對話歷史以避免重覆回覆
    clear_conversation_history()

    # 將新的回覆添加到對話歷史
    conversation_history.append({"role": "user", "content": answer})
    conversation_history.append({"role": "assistant", "content": response.choices[0].message.content})

    return response.choices[0].message.content

# 使用函數來獲取對話中的關鍵字
text = input("請輸入您的問題：")
print("關鍵字："+get_response(text))

while True:
    text = input()
    if text == "結束":
        clear_conversation_history()
        # 不需要重置 answer 變數，因為它在每次 get_response 被調用時都會被賦新值
        break
    print(get_response(text))

關鍵字：Keywords: 今天, 天氣, 很好
